# Customer Churn Prediction — Modeling & Tuning

This notebook focuses on training and optimizing machine learning models for telecom customer churn prediction.

Key objectives:
- Train an XGBoost baseline model
- Optimize decision threshold for business goals
- Compare with Logistic Regression and Random Forest
- Perform hyperparameter tuning
- Select the final production model

<h1>Business Objective</h1>

<h2>Business Context & Objective</h2>

<p>Customer churn represents a direct loss of recurring revenue for a telecom company. When a customer churns, the company loses not only their current monthly charges but also their expected future lifetime value. Additionally, customer acquisition costs already spent on that user are effectively wasted.</p>

There are two types of prediction errors to consider:

<b>False Positive (Type I Error)</b><br>
The model predicts that a customer will churn, but the customer would have stayed.
Business impact: The company may offer unnecessary discounts or retention incentives. This leads to moderate financial cost but does not result in customer loss.

<b>False Negative (Type II Error)</b><br>
The model predicts that a customer will not churn, but the customer actually churns.
Business impact: The company loses the customer’s recurring revenue and long-term lifetime value. This represents a significantly higher financial loss.

Since missing a true churner is more costly than offering an unnecessary incentive, the primary objective is to minimize False Negatives.

<b>Primary Metric:</b> Recall (maximize detection of actual churners)<br>
<b>Secondary Metric:</b> ROC-AUC (overall ranking performance)

In [13]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from sklearn.model_selection import RandomizedSearchCV

## 1. Load Processed Dataset
We load the cleaned and encoded dataset generated in previous notebooks.

In [2]:
df = pd.read_csv("../data/processed/telco_final_processed.csv")

In [3]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

## 2. Train-Test Split

The dataset is split into training and testing sets using stratified sampling to preserve churn distribution.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

## 3. Feature Scaling

StandardScaler is applied to normalize feature distributions.  
The scaler is fit only on the training set to prevent data leakage.

In [5]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## 4. Baseline XGBoost Model

We train a baseline XGBoost model before tuning.  
Class imbalance is handled using `scale_pos_weight`.

In [6]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=3,
    scale_pos_weight=(len(y_train) - sum(y_train)) / sum(y_train),
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

xgb_model.fit(X_train, y_train)

d:\Projects\Churn Analysis\venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [13:51:29] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

## 5. Baseline Model Evaluation

We evaluate the model using:
- Classification Report
- ROC-AUC
- Confusion Matrix

In [7]:
y_pred = xgb_model.predict(X_test)
y_prob = xgb_model.predict_proba(X_test)[:, 1]

print("Classification Report:\n")
print(classification_report(y_test, y_pred))

print("Test ROC-AUC:", roc_auc_score(y_test, y_prob))
print("Train ROC-AUC:", roc_auc_score(y_train, xgb_model.predict_proba(X_train)[:,1]))

print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Classification Report:

              precision    recall  f1-score   support

           0       0.91      0.73      0.81      1035
           1       0.52      0.80      0.63       374

    accuracy                           0.75      1409
   macro avg       0.71      0.76      0.72      1409
weighted avg       0.80      0.75      0.76      1409

Test ROC-AUC: 0.8404389160143634
Train ROC-AUC: 0.8909105086537148
Confusion Matrix:
 [[754 281]
 [ 75 299]]


XGBoost is:
- Stronger in recall than Logistic Regression likely was
- Probably competitive or slightly better than Random Forest
- Not severely overfitting

In [8]:
from sklearn.metrics import recall_score, precision_score

thresholds = np.arange(0.1, 0.9, 0.05)

results = []

for t in thresholds:
    y_pred_t = (y_prob >= t).astype(int)
    
    recall = recall_score(y_test, y_pred_t)
    precision = precision_score(y_test, y_pred_t)
    
    results.append((t, precision, recall))

results_df = pd.DataFrame(results, columns=["Threshold", "Precision", "Recall"])
results_df

,Threshold,Precision,Recall
0,0.10,0.353340,0.975936
1,0.15,0.376050,0.957219
2,0.20,0.399093,0.941176
3,0.25,0.420538,0.919786
4,0.30,0.446194,0.909091
5,0.35,0.460674,0.877005
6,0.40,0.478916,0.850267
7,0.45,0.494400,0.826203
8,0.50,0.515517,0.799465
9,0.55,0.536538,0.745989


## 6. Decision Threshold Optimization

Default classification threshold is 0.5.  
However, our business objective prioritizes **recall** to minimize missed churners.

We analyze different thresholds to balance precision and recall.

### Confusion Matrix at 0.35

In [9]:
# Apply new threshold
y_pred_035 = (y_prob >= 0.35).astype(int)

# Confusion Matrix
cm_035 = confusion_matrix(y_test, y_pred_035)
print("Confusion Matrix (Threshold = 0.35):\n")
print(cm_035)

# Classification Report
print("\nClassification Report (Threshold = 0.35):\n")
print(classification_report(y_test, y_pred_035))

Confusion Matrix (Threshold = 0.35):

[[651 384]
 [ 46 328]]

Classification Report (Threshold = 0.35):

              precision    recall  f1-score   support

           0       0.93      0.63      0.75      1035
           1       0.46      0.88      0.60       374

    accuracy                           0.69      1409
   macro avg       0.70      0.75      0.68      1409
weighted avg       0.81      0.69      0.71      1409



## 7. Model Comparison

We compare Logistic Regression, Random Forest, and XGBoost using the optimized threshold.

In [10]:
log_model = joblib.load("../models/logistic_model.pkl")
rf_model = joblib.load("../models/rf_model.pkl")

In [11]:
log_prob = log_model.predict_proba(X_test)[:, 1]
rf_prob = rf_model.predict_proba(X_test)[:, 1]

d:\Projects\Churn Analysis\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
d:\Projects\Churn Analysis\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [12]:
from sklearn.metrics import recall_score, precision_score, roc_auc_score

threshold = 0.35

models = {
    "Logistic": log_prob,
    "Random Forest": rf_prob,
    "XGBoost": y_prob
}

for name, probs in models.items():
    preds = (probs >= threshold).astype(int)
    
    print(f"\n{name}")
    print("Recall:", recall_score(y_test, preds))
    print("Precision:", precision_score(y_test, preds))
    print("ROC-AUC:", roc_auc_score(y_test, probs))


Logistic
Recall: 0.8636363636363636
Precision: 0.4318181818181818
ROC-AUC: 0.8269136376553256

Random Forest
Recall: 0.6657754010695187
Precision: 0.5545657015590201
ROC-AUC: 0.8276705675682656

XGBoost
Recall: 0.8770053475935828
Precision: 0.4606741573033708
ROC-AUC: 0.8404389160143634


## 8. Hyperparameter Tuning

RandomizedSearchCV is used to explore optimal XGBoost hyperparameters.

Search parameters include:
- tree depth
- learning rate
- number of estimators
- subsampling ratios

In [14]:
param_grid = {
    "n_estimators": [200, 300, 400],
    "max_depth": [3, 4, 5, 6],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5]
}

In [15]:
xgb_base = XGBClassifier(
    scale_pos_weight=(len(y_train) - sum(y_train)) / sum(y_train),
    random_state=42,
    use_label_encoder=False,
    eval_metric="logloss"
)

In [16]:
random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_grid,
    n_iter=25,
    scoring="roc_auc",
    cv=5,
    verbose=1,
    n_jobs=-1,
    random_state=42
)

random_search.fit(X_train, y_train)

Fitting 5 folds for each of 25 candidates, totalling 125 fits


d:\Projects\Churn Analysis\venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [16:17:11] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBClassifier...ree=None, ...)"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'colsample_bytree': [0.7, 0.8, ...], 'learning_rate': [0.01, 0.05, ...], 'max_depth': [3, 4, ...], 'min_child_weight': [1, 3, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",25
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be us

In [17]:
print("Best Params:", random_search.best_params_)
print("Best CV ROC-AUC:", random_search.best_score_)

Best Params: {'subsample': 0.7, 'n_estimators': 400, 'min_child_weight': 1, 'max_depth': 4, 'learning_rate': 0.01, 'colsample_bytree': 1.0}
Best CV ROC-AUC: 0.8481913582927794


In [18]:
# Get best tuned model
xgb_tuned = random_search.best_estimator_

# Predict probabilities
y_prob_tuned = xgb_tuned.predict_proba(X_test)[:, 1]

# ROC-AUC
test_roc_auc = roc_auc_score(y_test, y_prob_tuned)
print("Tuned Test ROC-AUC:", test_roc_auc)

# Apply chosen business threshold
threshold = 0.35
y_pred_tuned = (y_prob_tuned >= threshold).astype(int)

# Classification Report
print("\nClassification Report (Threshold = 0.35):\n")
print(classification_report(y_test, y_pred_tuned))

# Confusion Matrix
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_tuned))

Tuned Test ROC-AUC: 0.8461946834069596

Classification Report (Threshold = 0.35):

              precision    recall  f1-score   support

           0       0.94      0.60      0.73      1035
           1       0.44      0.89      0.59       374

    accuracy                           0.68      1409
   macro avg       0.69      0.74      0.66      1409
weighted avg       0.81      0.68      0.69      1409


Confusion Matrix:

[[620 415]
 [ 42 332]]


In [ ]:
comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest", "XGBoost"],
    "Recall": [0.864, 0.666, 0.877],
    "Precision": [0.432, 0.555, 0.461],
    "ROC-AUC": [0.827, 0.828, 0.840]
})

comparison

## 9. Final Model Selection

After hyperparameter tuning, XGBoost produced the best performance.

Final configuration:

Model: Tuned XGBoost  
Decision Threshold: 0.35  
Test ROC-AUC: 0.846  
Recall: 0.89  

This configuration prioritizes identifying high-risk churn customers while maintaining acceptable precision.

In [19]:
joblib.dump(xgb_tuned, "../models/xgb_tuned_model.pkl")

['../models/xgb_tuned_model.pkl']